## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [2]:
load_dotenv(override=True)

True

In [3]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 20

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [4]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "I want to become an independent insurance agent in Mexico (part-time, evenings 5-10pm, Direct official recruitment/application links for insurance agent training programs from Qualitas, Mapfre, and Inbursa in Mexico, specifically for Monterrey, Nuevo León, or their nearest regional office, find the links and send them to me"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [5]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [6]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

Here are the direct official links I found for insurance agent recruitment/training in Mexico:

- **Quálitas – “Conviértete en Agente”**: official application/contact form with fields for city/state/office selection, plus training and certification support. ([qualitas.com.mx](https://www.qualitas.com.mx/web/qmx/conviertete-en-agente?utm_source=openai))  
  Link: **https://www.qualitas.com.mx/web/qmx/conviertete-en-agente**

- **MAPFRE México – Solicitudes (oficina)**: official request portal for agents/alliados. I also found MAPFRE’s CTO directory showing a Monterrey-area office in **San Pedro Garza García, Nuevo León**, which appears to be the nearest regional office. ([zonaliados.mapfre.com.mx](https://zonaliados.mapfre.com.mx/Solicitud/principal?utm_source=openai))  
  Link: **https://zonaliados.mapfre.com.mx/Solicitud/principal**

- **Inbursa – Asesores portal**: official advisor portal with application/onboarding and training-related instructions, including contracting steps and documentation. ([portalasesores.inbursa.com](https://portalasesores.inbursa.com/sites/Asesores/home?utm_source=openai))  
  Link: **https://portalasesores.inbursa.com/sites/Asesores/home**

If you want, I can next look for the **exact Monterrey/Nuevo León contact page, phone number, or office address** for each one and narrow it down to the best part-time evening route.

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [7]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [8]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [9]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [10]:

result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find official recruitment or training application page for Quálitas insurance agent opportunities in Mexico, with Monterrey/Nuevo León relevance.', query='site:qualitas.com.mx agente de seguros reclutamiento capacitación Monterrey Nuevo León Quálitas oficial'), WebSearchItem(reason='Locate official Quálitas career or agent program page that may include regional offices or application links.', query='site:qualitas.com.mx bolsa de trabajo agente Monterrey Quálitas oficial'), WebSearchItem(reason='Find official Quálitas contact or office page for Monterrey/Nuevo León to identify nearest regional office for agent training/application.', query='site:qualitas.com.mx Monterrey Nuevo León sucursal Quálitas contacto oficial'), WebSearchItem(reason='Find official MAPFRE Mexico recruitment or training page for insurance agents, ideally with Monterrey or regional office info.', query='site:mapfre.com.mx agente de seguros reclutamiento capacitación Mont

## Agent 3: The Writer Agent

In [11]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## Agent 4: The email agent

In [12]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [13]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [14]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [15]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [16]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Showtime!

In [17]:
query = """I live in Monterrey, Nuevo León, Mexico. I want to become an independent insurance agent in Mexico (part-time, evenings 5-10pm) while keeping my full-time job as a Data Analyst. I have a sales background and enjoy building software tools (vibe-coding apps/systems).

Please find and list:
1. Direct official recruitment/application links for insurance agent training programs ("agente en formación" / "nuevo agente") from Qualitas, Mapfre, and Inbursa in Mexico, specifically for Monterrey, Nuevo León, or their nearest regional office.
2. Any other major Mexican insurance companies (GNP, AXA, HDI, Zurich, Chubb, Seguros Monterrey New York Life, Allianz) with similar new-agent formation programs, with direct links if available.
3. For each, note: whether they cover/reimburse the CNSF certification exam cost, whether the schedule can be flexible/part-time, and what requirements they list (education level, sales experience, etc.).

Prioritize official company pages over third-party job boards when possible, and flag if a company only recruits through job boards like OCC or Computrabajo rather than a dedicated careers page."""

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 20 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>